In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import boto3
import pickle
from pprint import pprint
import sklearn.metrics as skm
import warnings
# Suppress PerformanceWarning
warnings.filterwarnings('ignore')
try:
    import catboost as cb
except:
    ! pip install catboost
    import catboost as cb

### Functions

In [2]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # init client
    cls_client = boto3.client(
        's3',
    )
    # download file
    cls_client.download_file(
        str_project, 
        str_bucket_path, 
        str_local_path,
    )

### Constants

In [3]:
str_dirname_output = './output'

str_variant = 'noPTImodel10'

str_id = 'bigaccountid__app'

### Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Import stuff for preprocessing

In [5]:
list_str_filename = [
    'preprocessing.py',
    'cls_model_preprocessing.pkl',
]
for str_filename in tqdm(list_str_filename):
    # download
    str_bucket_path = f'01_ad/02_model/{str_variant}/00_preprocessing/01_create_preprocessor/{str_filename}'
    str_local_path = f'./{str_filename}'
    download_from_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project='20231010-gen-xii',
    )
# import
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))

100%|██████████| 2/2 [00:00<00:00,  6.28it/s]


### Import data

In [6]:
list_str_df = [
    'train',
    'valid',
    'test',
]
list_df = []
for str_df in tqdm(list_str_df):
    str_filename = f'df_{str_df}_raw.gzip'
    str_uri = f's3://20231010-gen-xii/02_pricing_pd/01_data_prep/03_train_valid_test_split/{str_filename}'
    df = pd.read_parquet(str_uri)
    df['data_set'] = str_df
    list_df.append(df)
# concatenate
df = pd.concat(list_df)
# get id
list_id = list(df[str_id])
# drop it
df.drop(str_id, axis=1, inplace=True)
# show
df

100%|██████████| 3/3 [00:04<00:00,  1.66s/it]


,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,RunningNetLoss,bitTarget24Months,target,data_set
38325,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,NaN,...,0.052130,1,1.106635,auto,0,2012-06-18 09:31:54.393,9070.52,1,1,train
38324,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,NaN,...,0.052130,1,1.106635,auto,0,2012-06-18 09:31:54.393,9070.52,1,1,train
38326,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,NaN,...,0.153369,1,1.254334,auto,1,2012-03-06 16:36:21.623,0.00,0,0,train
38327,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,NaN,...,0.153369,1,1.254334,auto,1,2012-03-06 16:36:21.623,0.00,0,0,train
38329,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,NaN,...,0.050115,0,1.249982,auto,0,2009-10-15 16:06:40.767,10250.98,1,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7864,20191231.0,20191231.0,NaN,201910.0,920826.0,2019-12-31,1.0,AU,2020-01-02,16196.0,...,0.068108,1,1.210836,auto,1,2009-12-28 16:59:42.413,0.00,0,0,test
7863,20191231.0,20191231.0,NaN,201910.0,920826.0,2019-12-31,1.0,AU,2020-01-02,16196.0,...,0.068108,1,1.210836,auto,1,2009-12-28 16:59:42.413,0.00,0,0,test
20360,20191231.0,20191231.0,NaN,201910.0,943817.0,2019-12-31,1.0,AU,2019-12-31,19634.0,...,0.077949,1,0.862106,auto,1,2019-10-16 09:37:21.113,0.00,0,0,test
20361,20191231.0,20191231.0,NaN,201910.0,943817.0,2019-12-31,1.0,AU,2019-12-31,19634.0,...,0.077949,1,0.862106,auto,1,2019-10-16 09:37:21.113,0.00,0,0,test


### Get the non-leaky features

In [7]:
str_filename = 'df_test_noleaks.gzip'
str_uri = f's3://20231010-gen-xii/02_pricing_pd/01_data_prep/05_leaky_features/04_write_dfs/{str_filename}'
list_cols = list(pd.read_parquet(str_uri).columns)
print(f'There are {len(list_cols)} non-leaky features')
# get those that are in df
list_cols = [col for col in list_cols if col in list(df.columns)]
print(f'There are {len(list_cols)} non-leaky features in df')

There are 1569 non-leaky features
There are 1568 non-leaky features in df


### Subset to non-leaky features

In [8]:
df = df[list_cols].copy()
df

,linkf060__tu,linkf045__tu,linkf185__tu,linkf193__tu,linkf105__tu,linkf195__tu,linkf032__tu,linkf083__tu,linkf084__tu,linkf078__tu,...,inttermbump__app,bitdealertrack__app,bitrouteone__app,bigdealertypeid__app,intservicecontractmileageaddon__app,bitmaintenanceagreement__app,uniqueid,applicationdate__app,target,data_set
38325,0.0,0.0,152.0,40.0,40.0,1141.0,N,0.0,1.0,0.0,...,6.0,1.0,0.0,2.0,24000.0,0.0,1.337511e+14,2013-10-01 07:28:54.833,1,train
38324,0.0,0.0,152.0,40.0,40.0,1141.0,N,0.0,1.0,0.0,...,6.0,1.0,0.0,2.0,24000.0,0.0,1.337511e+14,2013-10-01 07:28:54.833,1,train
38326,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,...,6.0,1.0,0.0,1.0,0.0,0.0,1.337528e+14,2013-10-01 08:25:39.117,0,train
38327,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,...,6.0,1.0,0.0,1.0,0.0,0.0,1.337528e+14,2013-10-01 08:25:39.117,0,train
38329,NaN,0.0,5.0,5.0,5.0,1802.0,N,1.0,1.0,11.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.337539e+14,2013-10-01 08:39:13.390,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7864,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,...,0.0,1.0,0.0,1.0,0.0,0.0,4.812499e+14,2019-12-31 16:09:44.410,0,test
7863,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,...,0.0,1.0,0.0,1.0,0.0,0.0,4.812499e+14,2019-12-31 16:09:44.410,0,test
20360,0.0,0.0,1182.0,1182.0,1182.0,1182.0,N,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,100000.0,0.0,4.812504e+14,2019-12-31 16:12:18.437,0,test
20361,0.0,0.0,1182.0,1182.0,1182.0,1182.0,N,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,100000.0,0.0,4.812504e+14,2019-12-31 16:12:18.437,0,test


### Preprocess data

In [9]:
df = cls_model_preprocessing.transform(df)
# rm
list_str_filename = [
    'cls_model_preprocessing.pkl',
    'preprocessing.py',
]
for str_filename in tqdm(list_str_filename):
    str_local_path = f'./{str_filename}'
    os.remove(str_local_path)
# assign id
df['bigAccountId'] = list_id
# show
df

NaN Replacer: 0.89643 sec.


100%|██████████| 3/3 [00:00<00:00, 33.25it/s]

Unable to convert vehiclemodel__app to string, not found in data
Set strings: 0.092216 sec.


Boolean Replacer: 2.7818 sec.


100%|██████████| 1558/1558 [00:01<00:00, 853.81it/s]


...

Replace inf and -inf with NaN: 3.0229 sec.
Imputer: 2.1266 sec.
Map term: 0.09824 sec.
Map PTI: 0.10957 sec.


100%|██████████| 9/9 [00:00<00:00, 894.01it/s]


Round values: 0.01272 sec.
Preprocessing Model: 17.817 sec.


100%|██████████| 2/2 [00:00<00:00, 4211.15it/s]


,linkf060__tu,linkf045__tu,linkf185__tu,linkf193__tu,linkf105__tu,linkf195__tu,linkf032__tu,linkf083__tu,linkf084__tu,linkf078__tu,...,data_set,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age,bigAccountId
38325,0.0,0.0,152.0,40.0,40.0,1141.0,n,0.0,1.0,0.0,...,train,2013,1.256262,10,4,0.09,1.370370,4.0,1.284932,1337511
38324,0.0,0.0,152.0,40.0,40.0,1141.0,n,0.0,1.0,0.0,...,train,2013,1.256262,10,4,0.09,1.370370,4.0,1.284932,1337511
38326,0.0,0.0,0.0,0.0,0.0,0.0,nan,0.0,0.0,0.0,...,train,2013,1.256262,10,4,0.15,1.586207,2.0,1.569863,1337528
38327,0.0,0.0,0.0,0.0,0.0,0.0,nan,0.0,0.0,0.0,...,train,2013,1.256262,10,4,0.15,1.586207,2.0,1.569863,1337528
38329,0.0,0.0,5.0,5.0,5.0,1802.0,n,1.0,1.0,11.0,...,train,2013,1.256262,10,4,0.03,1.342857,1.0,3.961644,1337539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7864,0.0,0.0,0.0,0.0,0.0,0.0,nan,0.0,0.0,0.0,...,test,2019,1.144717,12,4,0.09,1.548387,2.0,10.010959,4812498
7863,0.0,0.0,0.0,0.0,0.0,0.0,nan,0.0,0.0,0.0,...,test,2019,1.144717,12,4,0.09,1.548387,2.0,10.010959,4812498
20360,0.0,0.0,1182.0,1182.0,1182.0,1182.0,n,0.0,0.0,0.0,...,test,2019,1.144717,12,4,0.12,1.044444,-1.0,0.208219,4812503
20361,0.0,0.0,1182.0,1182.0,1182.0,1182.0,n,0.0,0.0,0.0,...,test,2019,1.144717,12,4,0.12,1.044444,-1.0,0.208219,4812503


### Get the features in the model

In [10]:
str_filename = 'final_model.pkl'
str_bucket_path = f'02_pricing_pd/02_model/{str_variant}/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project='20231010-gen-xii',
)
# import
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
# rm
os.remove(str_local_path)
# get features
list_cols_model = list(cls_model_inference.feature_names_)
# add id
list_cols = ['bigAccountId', 'data_set'] + list_cols_model

### Subset

In [11]:
df = df[list_cols].copy()
# show
df

,bigAccountId,data_set,fltgrossmonthly__income_sum,intopenbktype__app,bankruptcystatus__ln,bankruptcycount24month__ln,strname__app,strdealershiptrackertype__app,g990s__tu,bankruptcytimenewest__ln,...,agg903__tu,ret205__tu,at25s__tu,rev315__tu,g206b__tu,st25s__tu,businessassociationtimeoldest__ln,g218d__tu,ENG-loan_to_value,fltapproveddowntotal__app
38325,1337511,train,5000.0,nan,0.0,0.0,ohio,franchise,7.0,-1.0,...,0.0,-1.00,1.0,-1.0,0.000000,-1.0,-1.0,-1.0,1.370370,2500.0
38324,1337511,train,5000.0,nan,0.0,0.0,ohio,franchise,7.0,-1.0,...,0.0,-1.00,1.0,-1.0,0.000000,-1.0,-1.0,-1.0,1.370370,2500.0
38326,1337528,train,2000.0,7.0,1.0,1.0,tennessee,independent,2.0,4.0,...,1.0,8.02,1.0,7.0,0.000000,-1.0,-1.0,-1.0,1.586207,0.0
38327,1337528,train,2000.0,7.0,1.0,1.0,tennessee,independent,2.0,4.0,...,1.0,8.02,1.0,7.0,0.000000,-1.0,-1.0,-1.0,1.586207,0.0
38329,1337539,train,11000.0,nan,0.0,0.0,texas,independent,0.0,-1.0,...,0.0,-1.00,2.0,13.0,561.549063,1.0,-1.0,2.0,1.342857,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7864,4812498,test,5500.0,nan,0.0,0.0,georgia,franchise,-4.0,-1.0,...,0.0,-2.00,3.0,-1.0,374.322569,0.0,-1.0,-1.0,1.548387,0.0
7863,4812498,test,5500.0,nan,0.0,0.0,georgia,franchise,-4.0,-1.0,...,0.0,-2.00,3.0,-1.0,374.322569,0.0,-1.0,-1.0,1.548387,0.0
20360,4812503,test,4500.0,7.0,1.0,0.0,alabama,franchise,5.0,75.0,...,2.0,-1.00,4.0,2.0,0.000000,1.0,-1.0,0.0,1.044444,3500.0
20361,4812503,test,4500.0,7.0,1.0,0.0,alabama,franchise,5.0,75.0,...,2.0,-1.00,4.0,2.0,0.000000,1.0,-1.0,0.0,1.044444,3500.0


### Save

In [12]:
str_filename = 'df.gzip'
str_uri = f's3://20231010-gen-xii/14_monitoring/05_get_data/{str_filename}'
df.to_parquet(str_uri, compression='gzip')